# MSFT · Từ dữ liệu MBO đến điểm bất thường cấp phiên
Tháng 11/2020 dùng cho train, tháng 12 dành cho validation, hiệu chỉnh ngưỡng và test tiến về phía trước theo thời gian. Notebook chạy với Python 3.11+ trên máy cá nhân hoặc Google Colab. Trên Colab, mở notebook và chọn gói ZIP đi kèm ở ô đầu tiên. Gói này chứa đặc trưng của 42 phiên và kết quả đã đo. Có thể chọn GPU trước khi huấn luyện.

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, importlib.util
project = Path.cwd()
if (project / "msft_month_baseline" / "train.py").exists():
    project = project / "msft_month_baseline"
if not (project / "train.py").exists():
    archives = list(Path.cwd().glob("MSFT_month_baseline.zip"))
    if not archives:
        from google.colab import files
        print("Select MSFT_month_baseline.zip")
        uploaded = files.upload()
        archives = [Path(name) for name in uploaded if name.endswith(".zip")]
    archive, = archives
    with zipfile.ZipFile(archive) as z:
        z.extractall(Path.cwd())
    project = Path.cwd() / "msft_month_baseline"
assert (project/"train.py").is_file()
results = project / "results"
try:
    from compression import zstd
except ImportError:
    if importlib.util.find_spec("zstandard") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "zstandard"])
import numpy as np
import pandas as pd
from IPython.display import display, Image
print(project)

## 1. Tiền xử lý và phân chia
Các đặc trưng đã tính sẵn từ tệp gốc trên ổ D. Để tự dựng lại từ đầu, dùng thư mục kết quả mới và đặt đường dẫn Databento trong ô dưới. Pipeline kiểm tra SHA-256, dựng lại sổ lệnh và ghép SPY/QQQ theo phút vừa hoàn tất.

In [ ]:
REBUILD_RAW = False
DATA = Path("D:/DATA_AI4LIFE")
if REBUILD_RAW:
    results = project / "rebuild_results"
    subprocess.run([sys.executable, str(project/"prepare.py"), "--data", str(DATA),
                    "--out", str(results), "--workers", "2"], check=True)
audit = pd.read_csv(results/"sessions.csv").sort_values("day")
display(audit.groupby("split",sort=False).agg(first=("day","min"), last=("day","max"),
        sessions=("day","count"), expected_minutes=("minutes","sum"),
        usable_minutes=("usable","sum"), records=("records","sum")))

## 2. Phân tích đặc trưng trên tập train
Mỗi điểm dưới đây là trung vị theo phút kể từ lúc mở cửa qua các phiên train. Khối lượng khớp và số lượt hủy lệnh là tổng của một phút. Spread và độ sâu được lấy trung bình trong phút.

In [ ]:
import matplotlib.pyplot as plt
frames = []
for day in audit.loc[audit.split=="train","day"]:
    with np.load(results/"sessions"/f"{day}.npz") as d:
        frames.append(pd.DataFrame({k:d[k] for k in ["minute","spread","depth","cancel_count","trade_volume"]}))
training_profile = pd.concat(frames).groupby("minute").median()
training_profile.plot(subplots=True,layout=(2,2),figsize=(12,6),legend=True,grid=True)
plt.tight_layout()
plt.show()

## 3. Huấn luyện và chọn checkpoint
DHGNN mã hóa từng phút, GRU tích lũy trạng thái và EBM chấm diễn biến từ lúc mở cửa. Mỗi phiên train ghép với một bản đối chiếu có 30 phút bị trễ đồ thị lệnh. Hai chuỗi dùng cùng đặc trưng thị trường. Checkpoint có validation loss thấp nhất trong tối đa 15 epoch; dừng sớm với patience 4. Normalizer chỉ được fit trên tháng 11.

In [ ]:
subprocess.run([sys.executable, "-u", str(project/"train.py"), "--out", str(results)], check=True)
history = pd.read_csv(results/"history.csv")
display(history)
history.plot(x="epoch", y=["train_loss","validation_loss"],grid=True);

## 4. Hiệu chỉnh và đánh giá
Sau khi cố định checkpoint, lấy điểm cao nhất trong năm phiên hiệu chỉnh tại cùng phút kể từ mở cửa làm ngưỡng. Tác vụ giả lập gồm 12 ngày test quan sát và 12 bản gây trễ, một bản cho mỗi ngày. Sáu chỉ số phân loại chỉ mô tả tác vụ này. Với dữ liệu thật, bảng báo số cảnh báo và điểm, chưa có nhãn độc lập để tính độ chính xác phát hiện thao túng.

In [ ]:
import json
subprocess.run([sys.executable,str(project/"report.py"),"--out",str(results)],check=True)
metrics=json.loads((results/"metrics.json").read_text())
display(pd.Series(metrics["proxy_metrics"]).to_frame("value"))
display(pd.read_csv(results/"test_sessions.csv"))
display(Image(filename=str(results/"test_paths.png")))

## 5. Theo dõi một phiên từ mở cửa đến đóng cửa
Đổi ngày để xem năng lượng lũy kế, ngưỡng cảnh báo cùng thời điểm và lần đầu vượt ngưỡng. Ngày 24/12/2020 đóng cửa sớm.

In [ ]:
day = "2020-12-15"
scores = pd.read_csv(results/"test_intraday.csv")
selected = scores[scores.day==day]
display(selected.loc[selected.observed_alert,["window_end_ny","observed_energy","threshold"]].head(10))
selected.plot(x="elapsed_minutes", y=["observed_energy","threshold"], grid=True, figsize=(11,3));